In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.utils.data as data
from torchhd import functional, embeddings
from torchhd.datasets import EuropeanLanguages as Languages
import re, time, os

In [3]:
def set_seed(seed=123):
    import random, numpy as np
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(123)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [4]:
DIMENSIONS     = 10_000
MAX_INPUT_SIZE = 128
BATCH_SIZE     = 256
PADDING_IDX    = 0
PRINT_EVERY    = 100

ASCII_A = ord("a")
ASCII_Z = ord("z")
ASCII_SPACE = ord(" ")
NUM_TOKENS = (ASCII_Z - ASCII_A + 1) + 1 + 1  # letters + space + PAD slot

def char2int(char: str) -> int:
    a = ord(char)
    if a == ASCII_SPACE:
        return (ASCII_Z - ASCII_A + 1)
    if ASCII_A <= a <= ASCII_Z:
        return a - ASCII_A
    return (ASCII_Z - ASCII_A + 1)  # map non a–z to space

def transform(x: str) -> torch.Tensor:
    x = x.lower()
    x = re.sub(r"\s+", " ", x)
    x = x[:MAX_INPUT_SIZE]
    ids = [char2int(ch) + 1 for ch in x]  # shift by +1 so PAD is 0
    if len(ids) < MAX_INPUT_SIZE:
        ids += [PADDING_IDX] * (MAX_INPUT_SIZE - len(ids))
    return torch.tensor(ids, dtype=torch.long)


In [5]:
train_ds = Languages("/content/data", train=True,  transform=transform, download=True)
test_ds  = Languages("/content/data", train=False, transform=transform, download=True)

train_ld = data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=(DEVICE.type=="cuda"))
test_ld  = data.DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE.type=="cuda"))

len(train_ds), len(test_ds), train_ds.classes

Files already downloaded and verified
Files already downloaded and verified


(210032,
 21000,
 ['Bulgarian',
  'Czech',
  'Danish',
  'Dutch',
  'German',
  'English',
  'Estonian',
  'Finnish',
  'French',
  'Greek',
  'Hungarian',
  'Italian',
  'Latvian',
  'Lithuanian',
  'Polish',
  'Portuguese',
  'Romanian',
  'Slovak',
  'Slovenian',
  'Spanish',
  'Swedish'])

In [6]:
class Model(nn.Module):
    def __init__(self, num_classes, vocab_size, dim, padding_idx=0):
        super().__init__()
        self.symbol = embeddings.Random(vocab_size, dim, padding_idx=padding_idx)
        self.classify = nn.Linear(dim, num_classes, bias=False)
        with torch.no_grad():
            self.classify.weight.zero_()

    @torch.no_grad()
    def encode(self, x_ids: torch.Tensor) -> torch.Tensor:
        # We rely on TorchHD's ngrams (n=3) and hard_quantize, identical to the example.
        symbols = self.symbol(x_ids)                 # [B, T, D]
        hv = functional.ngrams(symbols, n=3)         # [B, D]
        hv = functional.hard_quantize(hv)            # sign -> {-1,+1}
        return hv

    def forward(self, x_ids: torch.Tensor) -> torch.Tensor:
        enc = self.encode(x_ids)                     # [B, D]
        return self.classify(enc)                    # [B, C]

model = Model(len(train_ds.classes), NUM_TOKENS, DIMENSIONS, padding_idx=PADDING_IDX).to(DEVICE)

In [9]:
import pdb
t0 = time.time()
with torch.no_grad():
    for bi, (samples, labels) in enumerate(train_ld, 1):
        samples = samples.to(DEVICE, non_blocking=True)
        labels  = labels.to(DEVICE, non_blocking=True)
        print('here', flush=True)
        print(labels, flush=True)
        print(samples, flush=True)
        samples_hv = model.encode(samples)                          # [B, D], bipolar
        model.classify.weight.index_add_(0, labels, samples_hv)     # accumulate into class rows
        if bi % PRINT_EVERY == 0:
            print(f"[train] {bi}/{len(train_ld)}")
            print(f"  |  elapsed: {time.time() - t0:.1f}s")

    # Normalize class rows (cosine-like scoring)
    model.classify.weight[:] = F.normalize(model.classify.weight, dim=1)
    print(f"Total Time Elapsed: {time.time() - t0:.1f}s")

here
tensor([ 4, 13,  4,  4,  1, 17, 15, 19, 14,  6,  1,  0, 11,  6, 17, 12,  9, 15,
        17, 11,  7, 17,  3,  1, 15,  3,  8,  9,  1, 14,  6, 16, 14, 19, 19,  3,
         3, 13, 15,  8,  9, 20, 15,  8,  5, 20, 16,  9, 19, 11,  5, 20,  0,  4,
         9, 10,  8, 18,  7, 15,  4,  8,  8,  8,  3, 18, 16,  1, 15,  7, 14,  5,
         7,  3, 15,  1, 14, 16, 18,  4,  9, 18, 13, 14,  9, 11, 17,  3, 17, 10,
         6,  7,  8, 19, 15, 15,  5,  1,  8, 17, 11, 19, 13,  8, 15, 15, 18,  4,
        19, 19,  7,  3,  6,  2, 12,  6,  2, 17, 18,  5, 17, 14,  0, 17, 13, 16,
         3, 11, 14, 16,  5, 13, 12,  8,  0,  7,  4,  1, 17, 12, 10,  7,  5, 14,
         7,  4,  0, 12,  2, 16,  2,  8, 16,  4,  6, 14,  5,  4, 14,  5, 20,  8,
        14, 15, 15, 15, 17, 10,  5, 12,  4, 11, 13, 12,  2, 12, 17, 18,  1,  5,
        15, 15, 10, 15, 19,  1,  1,  1,  6,  6, 18, 17, 18, 12, 13, 13,  7, 17,
        14,  0,  6, 17, 19, 16, 19, 10,  3,  3, 14, 19, 13,  4, 14, 13, 11,  1,
         3,  7, 16, 13,  1,  7,  2,

C:\Users\liang\AppData\Local\Temp\ipykernel_48476\1691475981.py:14: DeprecationWarning: torchhd.hard_quantize is deprecated, consider using torchhd.normalize instead.
  hv = functional.hard_quantize(hv)            # sign -> {-1,+1}


here
tensor([20,  5,  3, 10,  7, 13, 11, 20,  5, 14,  0, 14, 12,  4,  7,  0,  7, 16,
         5,  9, 10,  8, 17, 11,  7, 10, 14, 17, 20,  8,  4, 11, 20,  5,  4, 19,
         6, 20, 13,  2, 19,  3, 10,  2, 20, 13, 14,  1,  5,  7,  4, 12, 17,  2,
         9, 18, 19,  8,  5, 17,  3, 14, 13, 17, 16, 18,  2, 16, 16, 14, 16,  5,
        14, 20,  3, 10, 20,  7,  5,  4,  0,  9, 11, 12,  2,  2,  3, 17, 14, 18,
         8,  7,  2, 16,  6,  9,  7, 20, 11, 11,  4, 19, 13, 20, 14,  4,  8, 20,
        15, 20, 12, 11,  1,  1, 16,  4,  1,  5, 18, 12, 15,  7, 18,  4, 15, 15,
         9,  2, 19,  6,  2,  3, 15, 16,  4,  8, 10, 11, 20,  8,  8,  7, 18,  7,
        16,  6,  6,  8,  4, 13,  8,  8, 16,  6, 20,  8, 17,  5, 18,  5,  4,  4,
         6,  6, 19,  3, 20, 11,  5,  0,  6,  5,  2, 11, 15, 17,  0, 16,  1,  3,
        19,  4, 11, 10,  1,  1,  4, 14, 17, 19,  7, 11, 12,  0,  6,  6, 15,  5,
        15, 15, 18, 12,  2, 17,  6,  1,  5,  3, 14,  3, 18, 19,  1, 17,  1,  7,
         5,  7, 18, 15, 12, 17, 17,

KeyboardInterrupt: 